In [ ]:
!pip install requests chardet ipywidgets

In [12]:
on_button_click(None)

In [13]:
import io
import chardet
from IPython.display import display
import pandas as pd
import requests


def get_raw_bytes(source: str) -> bytes:
    """判斷是網址或本機路徑，並回傳 bytes"""
    source = source.strip().strip("'\"")

    if source.startswith("http://") or source.startswith("https://"):
        # 若為 GitHub 網頁連結，自動轉成 raw 資料網址
        if "github.com" in source and "/blob/" in source:
            source = source.replace("github.com", "raw.githubusercontent.com").replace(
                "/blob/", "/"
            )

        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(source, headers=headers, timeout=15)
        response.raise_for_status()
        return response.content
    else:
        with open(source, "rb") as f:
            return f.read()


def load_csv_with_fallback(data_bytes: bytes) -> pd.DataFrame:
    """依序嘗試常見中文編碼，失敗則使用 chardet 自動偵測"""
    candidate_encodings = ["utf-8-sig", "utf-8", "cp950", "big5"]

    for enc in candidate_encodings:
        try:
            df = pd.read_csv(io.BytesIO(data_bytes), encoding=enc)
            print(f"成功使用編碼讀取：【{enc}】")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue

    # 自動偵測編碼
    detected = chardet.detect(data_bytes[:20000])
    detected_enc = detected.get("encoding")
    print(f"一般編碼嘗試失敗，chardet 偵測為：【{detected_enc}】")

    if detected_enc:
        try:
            return pd.read_csv(io.BytesIO(data_bytes), encoding=detected_enc)
        except Exception:
            pass

    # 最終容錯
    print("注意：使用備用容錯模式 (errors='replace') 讀取")
    return pd.read_csv(
        io.BytesIO(data_bytes), encoding="utf-8", encoding_errors="replace"
    )


# ---------------- 執行區 ----------------
# 提示使用者輸入，輸入完直接按「Enter」即可執行
source_input = input("請輸入 CSV 網址或本機路徑 (按 Enter 確認): ").strip()

if not source_input:
    print("您沒有輸入任何路徑或網址！")
else:
    try:
        print(f"正在讀取資料：{source_input} ...")
        raw_data = get_raw_bytes(source_input)
        df = load_csv_with_fallback(raw_data)

        # 處理日期欄位：檢查是否有 Date 欄位，若無則預設第 1 欄為日期
        date_col = (
            "Date"
            if "Date" in df.columns
            else (df.columns[0] if len(df.columns) > 0 else None)
        )

        if date_col:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            print(f"已將欄位【{date_col}】轉換為日期格式 (Datetime)")

        print("\n讀取成功！前五列資料如下：")
        display(df.head(5))

    except FileNotFoundError:
        print("【錯誤】找不到檔案，請確認本機路徑是否正確（例如 D:/data.csv）。")
    except requests.exceptions.RequestException as e:
        print(f"【網路錯誤】下載失敗，請確認網址是否可公開存取：{e}")
    except Exception as e:
        print(f"【發生未預期錯誤】：{e}")

請輸入 CSV 網址或本機路徑 (按 Enter 確認):  https://raw.githubusercontent.com/tkme1981-hero/RFM-Project/refs/heads/main/SalesReport.csv


正在讀取資料：https://raw.githubusercontent.com/tkme1981-hero/RFM-Project/refs/heads/main/SalesReport.csv ...
成功使用編碼讀取：【utf-8-sig】
已將欄位【Date】轉換為日期格式 (Datetime)

讀取成功！前五列資料如下：


,Date,InvoNo,CustomerKey,Amount
0,2018-04-02,AP1803827,A1067,836.0
1,2018-04-03,AP1808235,A1129,836.0
2,2018-04-04,AP1807305,A1133,836.0
3,2018-04-05,AP1806145,A0464,836.0
4,2018-04-05,AP1803092,A0383,836.0


In [15]:
# 1. 確保日期欄位被正確解析（請將 'Date' 換成你 CSV 裡的實際欄位名）
df['Date'] = pd.to_datetime(df['Date'])

# 2. 以資料庫中的最大日期作為基準點，用來計算 Recency
max_date = df['Date'].max()

# 3. 核心轉換：利用 groupby 進行客戶別聚合計算
# （請根據你 CSV 實際的欄位名稱，修改 'CustomerKey'、'InvoNo'、'Amount'）
rfm_df = df.groupby('CustomerKey').agg({
    'Date': lambda x: (max_date - x.max()).days,  # Recency：最後一次交易離基準點幾天
    'InvoNo': 'nunique',                        # Frequency：獨特的發票編號總數
    'Amount': 'sum'                                # Monetary：Amount總加
}).reset_index()

# 4. 修改欄位名稱，讓表格一目了然
rfm_df.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# 5. 直接存在本機（存成無痛的 UTF-8 編碼，不帶索引欄位）
rfm_df.to_csv("RFM_Result.csv", index=False, encoding="utf-8-sig")

# 6. 秀出成果
rfm_df.head()

,CustomerID,Recency,Frequency,Monetary
0,A0350,9,25,51238.11440
1,A0351,16,6,17890.31200
2,A0352,14,14,46424.03852
3,A0353,24,14,27607.68636
4,A0354,31,6,9973.92064


In [18]:
rfm_df.to_csv("RFM_Data.csv",index=False)